In [11]:
import asyncio
import time
import json
from dotenv import load_dotenv
from pandas import unique


from app.services.external_api.searoute_api import SearouteApi
load_dotenv()

from app.services.db_service import DbService
sql_db_service = DbService()
await sql_db_service.init_pool()

searoute_api = SearouteApi("https://api.searoutes.com/", "EWhTo2x2hihNDCPZjCaMgFDWGegJoVLnYP7mqi5L")

In [27]:
import json

with open("./data/bubble_ports_raw.json") as f:
    ports_raw = json.loads(f.read())


(10297, 10297)

In [30]:
ports_with_mabux = [p for p in ports_raw if p.get("mabux_id") is not None or len(p.get("source_mabux_ids", [])) > 0 ]
len(ports_with_mabux)
ports_with_mabux[0]

{'Created By': 'admin_user_bunkering-53251_test',
 'search_key': 'a baiuca|esaxo|spain|',
 'Modified Date': '2024-08-30T12:45:38.048Z',
 'sr_country_name': 'Spain',
 'name': 'A Baiuca',
 'source_mabux_ids': ['539'],
 '_id': '1713117905273x207394668009318180',
 'Created Date': '2024-04-14T18:05:05.273Z',
 'locode': 'ESAXO',
 'sr_lon': -8.5,
 'sr_lat': 43.3}

In [23]:
import json

with open("./data/bubble_ports_raw.json") as f:
    all_ports = json.loads(f.read())

wrong_lat_ports = [p for p in all_ports if p['sr_lat'] is not None and abs(p['sr_lat']) > 90]
countryes = list(set([p['sr_country_name'] for p in wrong_lat_ports]))

KeyError: 'sr_lat'

In [24]:
all_ports[0]

{'Created By': 'admin_user_bunkering-53251_test',
 'search_key': 'a baiuca|esaxo|spain|',
 'Modified Date': '2024-08-30T12:45:38.048Z',
 'sr_country_name': 'Spain',
 'name': 'A Baiuca',
 'source_mabux_ids': ['539'],
 '_id': '1713117905273x207394668009318180',
 'Created Date': '2024-04-14T18:05:05.273Z',
 'locode': 'ESAXO',
 'sr_lon': -8.5,
 'sr_lat': 43.3}

In [13]:
found = []
not_found = []

for port in wrong_lat_ports:
    s_port, err = await searoute_api.get_port_info(port['locode'].strip())
    if s_port and not err:
        found.append(s_port)
    else:
        not_found.append(port)

    time.sleep(0.7)


In [17]:
with open("./data/updated_searoute_ports_with_correct_lat_and_lots.json", "w") as fp:
    json.dump([p.model_dump() for p in found], fp, indent=4)

In [18]:
r, err = await sql_db_service.bulk_upsert_ports(found)